
教学内容：封装 RAG 系统的检索功能，提供易用的 API
核心功能：文档加载与切片、向量生成与插入、多种检索方式（向量/关键字/混合）、
          可选 Rerank 重排序
前置知识：完成 03_retrieval_methods/ 的学习
后续学习：rag_qna_api.py（RAG 问答 API）
###### 


In [ ]:
import os
import sys
from dotenv import load_dotenv


def demo_basic_retrieval():
    """演示基础检索功能：初始化、建表、添加文档、检索"""
    print(f"\n-- 示例 1: 基础检索演示")

    # 初始化检索器
    retriever = RAGRetriever(
        milvus_uri=MILVUS_URI,
        collection_name="demo_retrieval",
        dim=DEFAULT_DIMENSION
    )

    # 创建 Collection
    retriever.create_collection()

    # 准备测试文档
    documents = [
        "机器学习是人工智能的核心技术，通过训练数据让计算机自动学习规律。",
        "深度学习使用多层神经网络，在图像识别和自然语言处理领域取得成功。",
        "自然语言处理让计算机理解和生成人类语言，应用包括机器翻译和智能客服。",
        "计算机视觉让计算机能够看懂图像，用于人脸识别和自动驾驶。",
        "推荐系统根据用户历史行为推荐相关内容，使用协同过滤算法。",
    ]

    # 添加文档
    retriever.add_documents(documents, chunk_size=100, chunk_overlap=20)

    # 检索测试
    print("\n检索测试：")

    queries = [
        "机器学习是什么？",
        "深度学习有什么应用？"
    ]

    for query in queries:
        print(f"\n查询：{query}")
        results = retriever.search(query, top_k=2)

        for i, result in enumerate(results):
            print(f"  [{i+1}] 相似度：{result['vector_score']:.4f}")
            print(f"      内容：{result['content'][:50]}...")

    retriever.close()


# =============================================================================
# 示例 2: 高级检索功能（混合检索 + Rerank）
# =============================================================================

In [ ]:
import os
import sys
from dotenv import load_dotenv


def demo_advanced_retrieval():
    """演示高级检索功能：纯向量检索 vs 带 Rerank 的检索"""
    print(f"\n-- 示例 2: 高级检索功能")

    retriever = RAGRetriever(
        milvus_uri=MILVUS_URI,
        collection_name="demo_advanced",
        dim=DEFAULT_DIMENSION
    )

    retriever.create_collection()

    documents = [
        "机器学习需要数学基础，包括线性代数、概率统计和微积分。",
        "深度学习是机器学习的子集，使用神经网络进行表征学习。",
        "Python 是 AI 开发的首选语言，有 TensorFlow 和 PyTorch 等框架。",
        "自然语言处理需要理解语法、语义和上下文信息。",
        "计算机视觉的核心任务包括分类、检测和分割。",
    ]

    retriever.add_documents(documents, chunk_size=100)

    print("\n1. 纯向量检索：")
    results = retriever.search("机器学习需要什么基础？", top_k=3, use_vector=True)
    for i, r in enumerate(results):
        print(f"  [{i+1}] {r['vector_score']:.3f} - {r['content'][:40]}...")

    print("\n2. 带 Rerank 的检索（需安装 FlagEmbedding）：")
    results = retriever.search("机器学习需要什么基础？", top_k=3, use_rerank=True)
    for i, r in enumerate(results):
        rerank = r.get('rerank_score', 'N/A')
        print(f"  [{i+1}] Rerank={rerank} - {r['content'][:40]}...")

    retriever.close()


# =============================================================================
# 示例 3: 从文件加载文档
# =============================================================================

In [ ]:
import os
import sys
from dotenv import load_dotenv


def demo_load_from_file():
    """演示从 TXT 文件加载文档并检索"""
    print(f"\n-- 示例 3: 从文件加载文档")

    retriever = RAGRetriever(
        milvus_uri=MILVUS_URI,
        collection_name="demo_file",
        dim=DEFAULT_DIMENSION
    )

    retriever.create_collection()

    # 从 TXT 文件加载
    file_path = os.path.join(os.path.dirname(__file__), "..", "data", "txt", "milvus_intro.txt")

    print(f"加载文件：{file_path}")

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        print(f"读取到 {len(content)} 字符")

        # 添加文档
        retriever.add_documents([content], chunk_size=200, chunk_overlap=50)

        # 检索测试
        print("\n检索测试：")
        results = retriever.search("Milvus 是什么？", top_k=2)

        for i, r in enumerate(results):
            print(f"  [{i+1}] {r['vector_score']:.3f}")
            print(f"      {r['content'][:60]}...")

    except FileNotFoundError:
        print(f"  文件不存在：{file_path}（请确保数据文件已准备）")

    retriever.close()


# =============================================================================
# 示例 4: API 参数详解
# =============================================================================

In [ ]:
import os
import sys
from dotenv import load_dotenv


def api_parameters_explained():
    """API 参数说明（纯文档展示）"""
    print(f"\n-- 示例 4: API 参数详解")

    print("""
RAGRetriever 参数说明
--------------------------------------------------------------

初始化参数：
  milvus_uri: Milvus 连接 URI
    - 远程服务："http://localhost:19530"
    - Milvus Lite（不支持 Windows）："milvus_demo.db"

  collection_name: Collection 名称
    - 默认："knowledge_base"

  embedding_model: Embedding 模型名称（本地 sentence-transformers）
    - 中文："bge-large-zh-v1.5"
    - 英文："all-MiniLM-L6-v2"
    - None: 使用阿里云 text-embedding-v4 API（默认）

  dim: 向量维度
    - 需与 Embedding 模型匹配
    - text-embedding-v4: 1024
    - bge-large-zh: 1024
    - MiniLM: 384

search() 参数：
  query: 查询文本
  top_k: 返回结果数量（建议 5-20）
  use_vector: 是否使用向量检索（默认 True）
  use_keyword: 是否使用关键字检索（默认 False）
  filter: 标量过滤条件
    - 示例："category == 'AI' and views > 1000"
  use_rerank: 是否使用 Rerank（默认 False）

方法总结：
  __init__          初始化 Milvus 连接
  create_collection 创建 Collection
  add_documents     添加文档（自动切片 + 向量化）
  search            检索（支持向量/关键字/Rerank）
  query             标量查询（按字段过滤）
  close             关闭连接
""")


# =============================================================================
# 主程序入口
# =============================================================================